# Concept Bottleneck Models on CLIP Features — with Test-Time Interventions
### Student version

**Course notebook · ~45–50 minutes (incl. exercises) · Google Colab (free T4 GPU is enough)**

A normal image classifier maps pixels straight to a label, and we have no idea *why* it said "Painted Bunting". A **Concept Bottleneck Model (CBM)** forces the prediction to pass through a layer of human-readable concepts (*"has red belly"*, *"has curved bill"*, ...). That buys us two things:

1. **Interpretability** — the label is a (here: linear) function of concepts we can read.
2. **Intervenability** — if a concept prediction is wrong, a human can *correct it at test time* and the downstream prediction updates.

In this notebook we build a CBM on top of **frozen CLIP image features** for the **CUB-200-2011** bird dataset, then study interventions in detail.

### Learning objectives
By the end you should be able to:
- explain the structure $x \rightarrow c \rightarrow y$ of a CBM and what "independent" vs "sequential" training means;
- build a CBM from frozen CLIP embeddings and CUB's attribute annotations;
- measure the *accuracy cost* of the bottleneck relative to a black-box linear probe;
- implement **group-wise concept interventions** and compare intervention *policies* (random / uncertainty / oracle);
- replace human concept labels by **CLIP zero-shot text–image similarity** and discuss what that gains and loses.

### Time plan
| Part | Content | Min |
|---|---|---|
| 0 | Setup, GPU check, **start the dataset download** | 4 |
| 1 | CUB concepts: loading, denoising, groups | 6 |
| 2 | Extract CLIP features (mostly waiting — read Part 3 meanwhile) | 5 |
| 3 | Black-box baseline vs. CBM (independent & sequential) | 9 |
| 4 | **Interventions**: curves, policies, a worked example | 12 |
| 5 | Label-free-ish concepts: CLIP zero-shot bottleneck | 6 |
| 6 | Wrap-up & discussion | 3 |

> **Before you start:** `Runtime → Change runtime type → T4 GPU`. Then run cells top to bottom.
>
> **✏️ Four short coding exercises** (look for `EXERCISE 1–4` in the code cells). Each is 1–3 lines and is followed by a self-check that tells you whether you got it right. "Run all" will stop at the first exercise you haven't filled in — that's intended.

## Background (read while the download runs)

Let $\phi$ be a frozen image encoder (here CLIP ViT-B/16). A CBM has two learned parts:

$$x \;\xrightarrow{\;\phi\;}\; z \;\xrightarrow{\;f\;}\; \hat c \in [0,1]^K \;\xrightarrow{\;g\;}\; \hat y$$

- **Concept predictor** $f$: predicts $K$ binary concepts (bird attributes) — trained with binary cross-entropy against concept labels $c$.
- **Label predictor** $g$: predicts the class *only* from the concepts. We use a linear $g$, so each class is a weighted sum of concepts.

Koh et al. (2020) distinguish how $g$ is trained:

| Variant | $g$ is trained on... | At test time $g$ sees... |
|---|---|---|
| **Independent** | ground-truth concepts $c$ | predicted $\hat c$ |
| **Sequential** | predicted concepts $\hat c$ (from a trained $f$) | predicted $\hat c$ |
| (Joint) | both trained end-to-end with a weighted loss | — *(not covered here)* |

**Intervention.** At test time, an expert replaces some entries of $\hat c$ with the correct value and we re-run $g$. CUB's 112 concepts come in **groups** (e.g. all the values of `has_belly_color` are one group), and a human is more naturally asked *"what colour is the belly?"* than *"is the belly red? is it blue? ..."*, so we intervene on whole groups.

## Part 0 — Setup

In [ ]:
!pip install -q open_clip_torch

In [ ]:
import os, time, random, urllib.request, tarfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import roc_auc_score
import open_clip

SEED = 0
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU")
if DEVICE == "cpu":
    print("WARNING: no GPU found. Feature extraction will take ~20+ min. "
          "Use Runtime > Change runtime type > T4 GPU.")

### Download CUB-200-2011 (~1.1 GB)
This tries the Caltech mirror first and falls back to a second mirror. It typically takes 1–3 minutes on Colab. **Run it now**, then read on.

*If both mirrors are blocked:* download `CUB_200_2011.tgz` manually, upload it to `/content/cub/`, and re-run this cell.

In [ ]:
DATA_DIR = Path("/content/cub")
MIRRORS = [
    "https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz?download=1",
    "https://s3.amazonaws.com/fast-ai-imageclas/CUB_200_2011.tgz",
]

def find_cub_root():
    hits = list(DATA_DIR.glob("**/attributes/image_attribute_labels.txt"))
    return hits[0].parent.parent if hits else None

if find_cub_root() is None:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    tgz = DATA_DIR / "CUB_200_2011.tgz"
    if not tgz.exists():
        for url in MIRRORS:
            try:
                print("Downloading", url)
                urllib.request.urlretrieve(url, tgz)
                break
            except Exception as e:
                print("  failed:", e)
                if tgz.exists(): tgz.unlink()
        else:
            raise RuntimeError("Could not download CUB. Upload CUB_200_2011.tgz to /content/cub/ and re-run.")
    print("Extracting ...")
    with tarfile.open(tgz) as t:
        t.extractall(DATA_DIR)
    tgz.unlink()

ROOT = find_cub_root()
print("CUB root:", ROOT)

## Part 1 — CUB concepts (~6 min)

CUB has 11,788 images of 200 bird species (≈30 train images per class in the official split) and **312 binary attributes** per image, annotated by crowd workers, grouped into 28 *groups* such as `has_bill_shape::curved_(up_or_down)`.

The raw per-image annotations are **noisy** (workers disagree, birds are partially hidden). We follow the preprocessing of Koh et al. (2020):

1. For each class, call an attribute *present* if it is present in **≥ 50 %** of that class's training images (**class-level majority vote**).
2. Keep only attributes that are present in **≥ 10 classes**. This leaves **112 concepts**.
3. Every image gets the *class-level* concept vector of its class.

Let's build this.

In [ ]:
# ---- metadata -------------------------------------------------------------
read = lambda f, names: pd.read_csv(ROOT / f, sep=" ", header=None, names=names)
images  = read("images.txt", ["img_id", "path"])
labels  = read("image_class_labels.txt", ["img_id", "cls"])
split   = read("train_test_split.txt", ["img_id", "is_train"])
classes = read("classes.txt", ["cls", "name"])

meta = images.merge(labels).merge(split).sort_values("img_id").reset_index(drop=True)
meta["cls"] = meta["cls"] - 1                                    # 0..199
meta["full_path"] = str(ROOT / "images") + "/" + meta["path"]
assert (meta.img_id.values == np.arange(1, len(meta) + 1)).all()

# attributes.txt sits either in ROOT or next to it, depending on the archive
attr_files = list(DATA_DIR.rglob("attributes.txt"))
assert attr_files, "attributes.txt not found - check the extracted archive"
attr_names = pd.read_csv(attr_files[0], sep=" ", header=None, names=["attr_id", "name"]).name.tolist()

print(f"{len(meta)} images | {meta.is_train.sum()} train / {(1 - meta.is_train).sum()} test "
      f"| {meta.cls.nunique()} classes | {len(attr_names)} raw attributes")

In [ ]:
# ---- per-image attribute matrix (N x 312) ----------------------------------
ATTR_FILE = ROOT / "attributes" / "image_attribute_labels.txt"

# Parse line by line: a few lines in this file are irregular, and pandas' fast parser
# would either crash or silently drop them. We only need the first three fields.
A_inst = np.zeros((len(meta), len(attr_names)), dtype=np.float32)
seen = np.zeros_like(A_inst, dtype=bool)
n_lines = 0
with open(ATTR_FILE) as f:
    for line in f:
        parts = line.split()
        if len(parts) < 3:
            continue
        i, a, v = int(parts[0]) - 1, int(parts[1]) - 1, int(parts[2])
        A_inst[i, a] = v
        seen[i, a] = True
        n_lines += 1
print(f"parsed {n_lines:,} annotations; coverage of the {A_inst.size:,} (image, attribute) cells: {seen.mean():.1%}")
assert seen.mean() > 0.99, "attribute file was not parsed correctly - check ATTR_FILE"

# ---- 1) class-level majority vote (train images only) -----------------------
cls_arr    = meta.cls.values
train_mask = meta.is_train.values == 1
class_freq = np.stack([A_inst[train_mask & (cls_arr == c)].mean(0) for c in range(200)])   # 200 x 312
class_attr = (class_freq >= 0.5).astype(np.float32)

# ---- 2) keep attributes present in >= 10 classes ----------------------------
keep = np.where(class_attr.sum(0) >= 10)[0]
concept_names = [attr_names[i] for i in keep]
K = len(concept_names)

# ---- 3) every image inherits its class's concept vector ---------------------
C_all = class_attr[cls_arr][:, keep]                                                       # N x K

# ---- concept groups (e.g. has_belly_color::*) -------------------------------
group_idx, group_labels = pd.factorize(np.array([n.split("::")[0] for n in concept_names]))
G = len(group_labels)

print(f"{K} concepts in {G} groups   (the standard CUB setup gives 112 concepts in 28 groups)")
if K < 100:
    print("WARNING: far fewer concepts than expected - check the attribute parsing above before continuing.")
print("avg. active concepts per image:", C_all.sum(1).mean().round(1))
print("distinct concept vectors across the 200 classes:", len(np.unique(class_attr[:, keep], axis=0)))
pd.Series(group_idx).map(dict(enumerate(group_labels))).value_counts().head(8)

In [ ]:
def class_name(c): return classes.name.iloc[c].split(".", 1)[1].replace("_", " ")

def show_example(i):
    row = meta.iloc[i]
    fig, ax = plt.subplots(figsize=(4, 3.5))
    ax.imshow(Image.open(row.full_path).convert("RGB")); ax.axis("off"); ax.set_title(class_name(row.cls))
    plt.show()
    present = [concept_names[k].replace("has_", "") for k in np.flatnonzero(C_all[i])]
    print(f"{len(present)} active concepts, e.g.:\n  " + "\n  ".join(present[:14]))

show_example(0)

**Checkpoint 1.** Note the line "distinct concept vectors across the 200 classes". Because every image inherits its class's concept vector, **concepts are a deterministic function of the class** here. Keep this in mind in Part 4 — what will happen when we feed *ground-truth* concepts to the label predictor?

**Your answer:** *(double-click this cell to write it)*

## Part 2 — Frozen CLIP features (~5 min)

We encode all 11,788 images **once** with CLIP ViT-B/16 and cache the L2-normalised 512-d embeddings. Everything after this point (concept predictor, label heads, interventions) is just small linear layers on cached features — which is why the rest of the notebook runs in seconds and why CLIP + CBM is such a convenient teaching setup.

*(Swap `MODEL_NAME` for `"ViT-B-32"` for ~2× faster extraction or `"ViT-L-14"` for stronger features.)*

In [ ]:
MODEL_NAME, PRETRAINED = "ViT-B-16", "openai"
clip_model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
clip_model = clip_model.to(DEVICE).eval()
tokenizer = open_clip.get_tokenizer(MODEL_NAME)

FEAT_FILE = Path(f"/content/cub_feats_{MODEL_NAME}_{PRETRAINED}.pt")

class ImageDS(torch.utils.data.Dataset):
    def __init__(self, paths, tf): self.paths, self.tf = list(paths), tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return self.tf(Image.open(self.paths[i]).convert("RGB"))

if FEAT_FILE.exists():
    X_raw = torch.load(FEAT_FILE)
    print("loaded cached features", tuple(X_raw.shape))
else:
    BS = 128
    dl = torch.utils.data.DataLoader(ImageDS(meta.full_path, preprocess), batch_size=BS,
                                     num_workers=min(4, os.cpu_count() or 2), pin_memory=(DEVICE == "cuda"))
    feats, t0 = [], time.time()
    with torch.no_grad(), torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=(DEVICE == "cuda")):
        for i, x in enumerate(dl):
            f = clip_model.encode_image(x.to(DEVICE, non_blocking=True))
            feats.append(F.normalize(f.float(), dim=-1).cpu())
            if i % 10 == 0:
                print(f"  {min((i + 1) * BS, len(meta))}/{len(meta)} images  ({time.time() - t0:.0f}s)")
    X_raw = torch.cat(feats)
    torch.save(X_raw, FEAT_FILE)
    print("done in %.0fs" % (time.time() - t0), tuple(X_raw.shape))

In [ ]:
# ---- splits: official train/test; hold out 20% of train for validation ------
TR_ALL = np.where(meta.is_train.values == 1)[0]
TE     = np.where(meta.is_train.values == 0)[0]
perm   = np.random.RandomState(SEED).permutation(TR_ALL)
n_val  = int(0.2 * len(perm))
VA, TR = np.sort(perm[:n_val]), np.sort(perm[n_val:])
tr, va, te = [torch.as_tensor(a, device=DEVICE) for a in (TR, VA, TE)]
print(f"train {len(TR)} | val {len(VA)} | test {len(TE)}")

# ---- standardise features with train statistics ------------------------------
mu, sd = X_raw[TR].mean(0, keepdim=True), X_raw[TR].std(0, keepdim=True)
X = ((X_raw - mu) / sd).to(DEVICE)                         # N x 512  (z-scored)
Y = torch.as_tensor(meta.cls.values, device=DEVICE)        # N        (class ids)
C = torch.as_tensor(C_all, device=DEVICE)                  # N x K    (ground-truth concepts)

## Part 3 — Black box vs. concept bottleneck (~9 min)

### 3.1 A tiny training helper
All our models are single linear layers. One helper trains any of them with minibatch AdamW and keeps the epoch with the best **validation** score.

In [ ]:
def fit_linear(Xtr, Ytr, Xva, Yva, out_dim, loss_fn, score_fn, epochs=100, lr=1e-3, wd=1e-4, bs=256, tag=""):
    set_seed(SEED)
    model = nn.Linear(Xtr.shape[1], out_dim).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    best, best_state, best_ep = -1e9, None, 0
    for ep in range(epochs):
        order = torch.randperm(len(Xtr), device=DEVICE)
        for i in range(0, len(Xtr), bs):
            idx = order[i:i + bs]
            loss = loss_fn(model(Xtr[idx]), Ytr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            s = score_fn(model(Xva), Yva).item()
        if s > best:
            best, best_ep = s, ep
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    print(f"[{tag}] best val score {best:.4f} (epoch {best_ep + 1}/{epochs})")
    return model.eval()

cls_score     = lambda logits, y: (logits.argmax(1) == y).float().mean()          # top-1 accuracy
concept_score = lambda logits, c: ((logits > 0).float() == c).float().mean()      # per-concept accuracy

@torch.no_grad()
def predict(head, Z): return head(Z).argmax(1)

@torch.no_grad()
def acc(head, Z, y): return (head(Z).argmax(1) == y).float().mean().item()

### 3.2 Baseline: black-box linear probe
CLIP features → 200 classes directly. No concepts, no explanation. This is the accuracy we are willing to *trade* for interpretability.

In [ ]:
probe = fit_linear(X[tr], Y[tr], X[va], Y[va], 200, F.cross_entropy, cls_score,
                   epochs=100, lr=1e-3, wd=1e-2, tag="black-box probe")
acc_probe = acc(probe, X[te], Y[te])
print(f"\nBlack-box linear probe — test accuracy: {acc_probe:.3f}")

### 3.3 Concept predictor $f$: CLIP features → 112 concepts
Binary cross-entropy against the ground-truth concept vector.

In [ ]:
concept_head = fit_linear(X[tr], C[tr], X[va], C[va], K, F.binary_cross_entropy_with_logits, concept_score,
                          epochs=100, lr=1e-3, wd=1e-2, tag="concept predictor f")
with torch.no_grad():
    P = torch.sigmoid(concept_head(X))                     # predicted concept probabilities, N x K

def per_concept_auc(scores, targets):
    s, t = scores.detach().cpu().numpy(), targets.detach().cpu().numpy()
    return np.array([roc_auc_score(t[:, k], s[:, k]) if 0 < t[:, k].sum() < len(t) else np.nan
                     for k in range(t.shape[1])])

auc_trained = per_concept_auc(P[te], C[te])
print(f"Test concept accuracy: {((P[te] > 0.5).float() == C[te]).float().mean().item():.3f}   "
      f"| mean per-concept AUC: {np.nanmean(auc_trained):.3f}")

### 3.4 Label predictor $g$: concepts → class  ✏️ *Exercise 1*
- **Independent**: train $g$ on *ground-truth* concepts; early-stop on val using *predicted* concepts.
- **Sequential**: train $g$ on the concept predictor's *outputs* for the training set.

In [ ]:
# ---- EXERCISE 1 ------------------------------------------------------------------
# Decide what each label head is TRAINED on (Xtr) and EARLY-STOPPED on (Xva).
# Building blocks: C = ground-truth concepts, P = predicted concept probabilities
# (both are N x K; index them with the split tensors tr / va).
Xtr_ind, Xva_ind = None, None     # TODO: independent head
Xtr_seq, Xva_seq = None, None     # TODO: sequential head
assert Xtr_ind is not None and Xtr_seq is not None, "Fill in the TODOs above (Exercise 1)"
assert Xtr_ind.shape == (len(tr), K) and Xva_seq.shape == (len(va), K), "wrong shapes - did you index with tr / va?"

head_ind = fit_linear(Xtr_ind, Y[tr], Xva_ind, Y[va], 200, F.cross_entropy, cls_score,
                      epochs=100, lr=1e-2, wd=1e-4, tag="g independent")
head_seq = fit_linear(Xtr_seq, Y[tr], Xva_seq, Y[va], 200, F.cross_entropy, cls_score,
                      epochs=100, lr=1e-2, wd=1e-4, tag="g sequential")

acc_ind, acc_seq = acc(head_ind, P[te], Y[te]), acc(head_seq, P[te], Y[te])
print(f"\nTest accuracy   black-box probe: {acc_probe:.3f} | CBM independent: {acc_ind:.3f} | CBM sequential: {acc_seq:.3f}")
print(f"Accuracy of the independent head when given the TRUE concepts: {acc(head_ind, C[te], Y[te]):.3f}")

### 3.5 Reading the model
Because $g$ is linear over named concepts, each class has a readable "recipe". Here are the highest-weighted concepts for a few classes:

In [ ]:
W = head_seq.weight.detach().cpu().numpy()                 # 200 x K
for c in [0, 9, 49]:                                       # a few classes
    top = np.argsort(-W[c])[:6]
    print(f"{class_name(c):28s} ->", ", ".join(concept_names[k].replace("has_", "") for k in top))

**Checkpoint 2.** (a) How large is the accuracy drop from the black box to the CBM? (b) Why might the *sequential* head be a bit more robust than the *independent* one on un-intervened inputs, even though the independent head sees "cleaner" training data?

**Your answer:** *(double-click this cell to write it)*

## Part 4 — Test-time interventions (~12 min)

### 4.1 Definition
For a test image and a set $S$ of concept **groups**, an intervention replaces the predicted probabilities of every concept in $S$ by the expert's value:

$$\hat c_j \;\leftarrow\; c_j \quad \text{for all concepts } j \text{ belonging to a group in } S,$$

then we recompute $\hat y = g(\hat c)$. Nothing is retrained.

### 4.2 Policies: *which* groups should we ask the expert about?
Expert time is limited, so the order matters. We compare three policies (each ranks the $G=28$ groups per test image):

| Policy | Rank groups by | Needs ground truth? |
|---|---|---|
| **random** | random order | no |
| **uncertainty** | summed entropy of the group's predicted concept probabilities (most uncertain first) | no — deployable |
| **oracle** | summed error $|\hat c - c|$ (most wrong first) | yes — an upper bound on any policy |

✏️ **Exercises 2 and 3** are in the next two code cells: implement the intervention itself, and the uncertainty measure behind the *uncertainty* policy.


In [ ]:
group_of = torch.as_tensor(group_idx, device=DEVICE).long()          # concept -> group id, length K

def group_ranks(policy, N, unc=None, err=None, seed=0):
    # returns rank[n, g]: the position (0 = first) at which group g is intervened on for sample n
    if policy == "random":
        gen = torch.Generator(device=DEVICE); gen.manual_seed(seed)
        score = torch.rand(N, G, device=DEVICE, generator=gen)
    else:
        per_concept = unc if policy == "uncertainty" else err
        score = torch.zeros(N, G, device=DEVICE).index_add_(1, group_of, per_concept)
    order = score.argsort(dim=1, descending=True)                  # groups, first-to-intervene first
    return order.argsort(dim=1)                                    # invert: group -> position

def intervene(Z, target, rank, k):
    # replace the concepts of the first k groups (per sample) by their target values
    # ---- EXERCISE 2 ----------------------------------------------------------
    # Build a boolean mask (N x K) that is True where a concept's group is among the
    # first k groups chosen for that sample.
    # Hint: rank is N x G (rank[n, g] = position of group g for sample n); group_of is length K.
    mask = None                                                    # TODO
    assert mask is not None, "Fill in the TODO (Exercise 2)"
    return torch.where(mask, target, Z)


# ---- self-check for Exercise 2 -------------------------------------------------------
_ar = torch.arange(G, device=DEVICE)
_rank = torch.stack([_ar, _ar.flip(0)])                         # sample 0: group 0 first | sample 1: group G-1 first
_Z, _T = torch.zeros(2, K, device=DEVICE), torch.ones(2, K, device=DEVICE)
_sizes = torch.bincount(group_of, minlength=G)
assert intervene(_Z, _T, _rank, 0).sum() == 0, "k=0 must change nothing"
assert intervene(_Z, _T, _rank, G).sum() == 2 * K, "k=G must change everything"
_one = intervene(_Z, _T, _rank, 1)
assert _one[0].sum() == _sizes[0] and _one[1].sum() == _sizes[G - 1], "k=1 should change exactly the concepts of the first group"
print("Exercise 2 passed")

In [ ]:
# -- precompute rankings on the test set (they depend only on P and C, not on the label head)
Pte, Cte, Yte = P[te], C[te], Y[te]
# ---- EXERCISE 3 ----------------------------------------------------------------------
# Per-concept uncertainty = binary entropy of the predicted probability, H(p) = -[p log p + (1-p) log(1-p)].
# Use Pte; add 1e-8 inside the logs for numerical stability.  Result: N_te x K tensor.
entropy = None                                                 # TODO
assert entropy is not None and entropy.shape == Pte.shape, "Fill in the TODO (Exercise 3)"
assert -1e-4 <= entropy.min().item() and entropy.max().item() <= np.log(2) + 1e-4, "binary entropy must lie in [0, ln 2]"
print("Exercise 3 passed")
N_te = len(Pte)
ranks = {
    "uncertainty": group_ranks("uncertainty", N_te, unc=entropy),
    "oracle":      group_ranks("oracle", N_te, err=(Pte - Cte).abs()),
}
random_ranks = [group_ranks("random", N_te, seed=s) for s in range(5)]    # average over 5 seeds

ks = list(range(G + 1))
def curve(head, Z, target, y, policy):
    if policy == "random":
        return np.mean([[acc(head, intervene(Z, target, r, k), y) for k in ks] for r in random_ranks], axis=0)
    return np.array([acc(head, intervene(Z, target, ranks[policy], k), y) for k in ks])

curves = {(h, p): curve(head, Pte, Cte, Yte, p)
          for h, head in [("independent", head_ind), ("sequential", head_seq)]
          for p in ["random", "uncertainty", "oracle"]}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, h in zip(axes, ["independent", "sequential"]):
    for p, style in [("random", "-"), ("uncertainty", "-"), ("oracle", "--")]:
        ax.plot(ks, curves[(h, p)], style, lw=2, label=p)
    ax.axhline(acc_probe, color="gray", ls=":", label="black-box probe")
    ax.set_title(f"CBM ({h})"); ax.set_xlabel("# concept groups intervened on (per image)")
    ax.grid(alpha=.3)
axes[0].set_ylabel("test accuracy"); axes[0].legend(loc="lower right")
plt.tight_layout(); plt.show()

rows = [{"head": h, "policy": p, **{f"k={k}": curves[(h, p)][k] for k in (0, 1, 3, 5, 10, G)}}
        for (h, p) in curves]
pd.DataFrame(rows).round(3)

**Checkpoint 3.** Look at the curves. (a) How many groups does the *uncertainty* policy need to beat the black-box probe, compared with *random*? (b) Why is the *independent* head typically steeper than the *sequential* one? (c) Why do both reach very high accuracy at $k=G$?

**Your answer:** *(double-click this cell to write it)*

### 4.3 A worked example: fixing one bird
Let's find a test image that the independent CBM gets wrong but that is fixed by intervening on just a few groups using the oracle policy, and look at *which* concepts were wrong.

In [ ]:
base_pred = predict(head_ind, Pte)
for k_demo in (2, 3, 5, 8, G):
    after = predict(head_ind, intervene(Pte, Cte, ranks["oracle"], k_demo))
    cand = ((base_pred != Yte) & (after == Yte)).nonzero().flatten().tolist()
    if cand: break

j = cand[min(7, len(cand) - 1)]                                   # position in the test set
row = meta.iloc[TE[j]]
flipped = (ranks["oracle"][j][group_of] < k_demo) & ((Pte[j] > 0.5).float() != Cte[j])

fig, ax = plt.subplots(figsize=(4.5, 3.8))
ax.imshow(Image.open(row.full_path).convert("RGB")); ax.axis("off")
ax.set_title(f"true: {class_name(row.cls)}", fontsize=10); plt.show()

print(f"BEFORE: predicted '{class_name(base_pred[j].item())}'  (wrong)")
print(f"Intervening on the {k_demo} most-wrong groups corrects {int(flipped.sum())} concept predictions:")
for k in torch.where(flipped)[0].tolist():
    print(f"   {concept_names[k].replace('has_', ''):45s} p̂ = {Pte[j, k].item():.2f}  ->  {int(Cte[j, k].item())}")
print(f"AFTER:  predicted '{class_name(after[j].item())}'  (correct)")

This is the practical promise of CBMs: instead of "the network is wrong, retrain it", a user can say **"no, the belly is yellow, not white"** and get a corrected answer immediately.

> **Try it (2 min).** Change `k_demo` candidates, or pick a different `j` (e.g. `cand[0]`, `cand[-1]`). Do the wrong concepts tend to be about *colour*, *pattern* or *shape*? We'll quantify this in Part 5.

## Part 5 — Concepts without concept labels: CLIP zero-shot (~6 min)

Annotating hundreds of attributes per image (as CUB did) is expensive. CLIP gives us a cheaper option: **score each concept by the similarity between the image embedding and a text prompt describing the concept**, e.g.

> "a photo of a bird whose belly color is yellow"

✏️ **Exercise 4** is in the next code cell: write the prompt template.

This is the idea behind *Label-free CBMs* (Oikarinen et al., 2023) and *Language in a Bottle* (Yang et al., 2023), simplified here: we reuse CUB's 112 concept *names* but ignore their *labels* when building the bottleneck. The concept scores are z-scored and a linear head maps them to classes — the concept layer needs **no concept supervision at all**.

In [ ]:
def concept_to_prompt(name):
    # ---- EXERCISE 4 -----------------------------------------------------------------
    # Turn a concept name such as "has_belly_color::yellow" into a natural-language prompt
    # for CLIP, e.g. "a photo of a bird whose belly color is yellow".
    grp, val = name.split("::")
    prompt = None                                              # TODO
    return prompt

_p = concept_to_prompt("has_belly_color::yellow")
assert _p and "belly color" in _p and "yellow" in _p and "has_" not in _p and "_" not in _p, "Exercise 4: got %r" % (_p,)
print("Exercise 4 passed:", _p)

prompts = [concept_to_prompt(n) for n in concept_names]
print("e.g.:", prompts[0], "|", prompts[K // 2], "|", prompts[-1])

with torch.no_grad():
    T = F.normalize(clip_model.encode_text(tokenizer(prompts).to(DEVICE)).float(), dim=-1)   # K x 512

S_raw = 100.0 * X_raw.to(DEVICE) @ T.T                              # N x K  image-text similarities
s_mu, s_sd = S_raw[tr].mean(0, keepdim=True), S_raw[tr].std(0, keepdim=True)
S = (S_raw - s_mu) / s_sd                                           # z-scored concept scores

auc_zs = per_concept_auc(S[te], C[te])
print(f"\nMean per-concept AUC  —  zero-shot CLIP: {np.nanmean(auc_zs):.3f}   |   trained concept predictor: {np.nanmean(auc_trained):.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(auc_trained, auc_zs, s=14, alpha=.7); ax[0].plot([0.3, 1], [0.3, 1], "k--", lw=1)
ax[0].set_xlabel("AUC — trained concept predictor"); ax[0].set_ylabel("AUC — CLIP zero-shot"); ax[0].set_title("Per-concept AUC (test)")

g_auc = (pd.DataFrame({"group": pd.Series(group_labels[group_idx]).str.replace("has_", "", regex=False), "zero-shot": auc_zs, "trained": auc_trained})
           .groupby("group").mean().sort_values("zero-shot"))
g_auc.plot.barh(ax=ax[1], width=.8, legend=True); ax[1].set_title("Mean AUC per concept group"); ax[1].tick_params(axis="y", labelsize=7)
plt.tight_layout(); plt.show()

In [ ]:
head_zs = fit_linear(S[tr], Y[tr], S[va], Y[va], 200, F.cross_entropy, cls_score,
                     epochs=100, lr=1e-3, wd=1e-2, tag="g on zero-shot concepts")
acc_zs = acc(head_zs, S[te], Y[te])
print(f"\nTest accuracy — zero-shot concept bottleneck: {acc_zs:.3f}  "
      f"(black-box {acc_probe:.3f} | CBM-seq {acc_seq:.3f})")

### Can we intervene on the zero-shot bottleneck too?
Yes, but the concept scores are uncalibrated z-scores, not probabilities. A simple fix: when an expert says concept $j$ is present (absent), set its score to the **average z-score of positive (negative) training images** for that concept. We can only use the *random* and *oracle* policies here, since the zero-shot scores come with no probability to compute entropy from.

In [ ]:
pos = C[tr]
v1 = (S[tr] * pos).sum(0) / pos.sum(0).clamp(min=1)                 # mean score when concept present
v0 = (S[tr] * (1 - pos)).sum(0) / (1 - pos).sum(0).clamp(min=1)     # mean score when absent
Sz_te = S[te]
tgt_zs = torch.where(Cte == 1, v1, v0)                              # N x K  intervention targets

ranks_zs = {"oracle": group_ranks("oracle", N_te, err=(Sz_te - tgt_zs).abs())}
rand_zs  = [group_ranks("random", N_te, seed=s) for s in range(5)]
zs_random = np.mean([[acc(head_zs, intervene(Sz_te, tgt_zs, r, k), Yte) for k in ks] for r in rand_zs], axis=0)
zs_oracle = np.array([acc(head_zs, intervene(Sz_te, tgt_zs, ranks_zs["oracle"], k), Yte) for k in ks])

plt.figure(figsize=(6.5, 4))
plt.plot(ks, curves[("sequential", "random")],  "C0-",  label="trained CBM (seq) — random")
plt.plot(ks, curves[("sequential", "oracle")],  "C0--", label="trained CBM (seq) — oracle")
plt.plot(ks, zs_random, "C3-",  label="zero-shot bottleneck — random")
plt.plot(ks, zs_oracle, "C3--", label="zero-shot bottleneck — oracle")
plt.axhline(acc_probe, color="gray", ls=":", label="black-box probe")
plt.xlabel("# concept groups intervened on"); plt.ylabel("test accuracy"); plt.grid(alpha=.3); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Part 6 — Wrap-up (~3 min)

In [ ]:
summary = pd.DataFrame([
    {"model": "black-box linear probe",             "needs concept labels": "—",   "test acc": acc_probe, "acc @ all groups corrected": np.nan},
    {"model": "CBM independent (CLIP + CUB attrs)", "needs concept labels": "yes", "test acc": acc_ind,   "acc @ all groups corrected": curves[("independent", "oracle")][-1]},
    {"model": "CBM sequential  (CLIP + CUB attrs)", "needs concept labels": "yes", "test acc": acc_seq,   "acc @ all groups corrected": curves[("sequential", "oracle")][-1]},
    {"model": "zero-shot CLIP bottleneck",          "needs concept labels": "no",  "test acc": acc_zs,    "acc @ all groups corrected": zs_oracle[-1]},
]).round(3)
summary

### Discussion questions
1. **Is the bottleneck really "interpretable"?** The concept layer holds *probabilities*, which can carry extra information about the image beyond the named concept (so-called *concept leakage*). How could you test whether the label head is using more than the concept's meaning? *(Hint: what happens if you threshold the concepts to 0/1 before the label head?)*
2. **Intervention realism.** Our "expert" is an oracle with access to denoised, class-level labels. What would change with a human who is wrong 10–20 % of the time? How would you simulate that?
3. **Concept set design.** CUB's 112 concepts were chosen for birds by experts. What breaks if the concept set is *incomplete* (some distinguishing feature isn't covered)? Compare with the zero-shot bottleneck where you can add concepts for free.
4. **CLIP as a concept scorer.** Which groups did zero-shot CLIP do poorly on? Is that a limitation of CLIP's attribute understanding, of our prompt template, or of the labels?
5. **Cost-benefit.** If an intervention costs 5 seconds of expert time and an error costs much more, what operating point on the curve would you choose?

### Optional extensions
- **Noisy expert:** flip 15 % of the target concepts before intervening and re-plot Part 4.
- **Concept-level interventions:** intervene on individual concepts (rank by entropy) instead of groups.
- **Better prompts:** write 3 paraphrases per concept and average the text embeddings; does zero-shot AUC improve?
- **Sparser explanations:** add an L1 penalty to the label head and see how many concepts per class survive.
- **Stronger backbone:** set `MODEL_NAME = "ViT-L-14"` and re-run everything.
- **Joint training:** train $f$ and $g$ end-to-end with loss $\mathcal{L}_y + \lambda\,\mathcal{L}_c$ and compare intervention behaviour.

### References
- Koh et al. *Concept Bottleneck Models.* ICML 2020.
- Radford et al. *Learning Transferable Visual Models From Natural Language Supervision (CLIP).* ICML 2021.
- Wah et al. *The Caltech-UCSD Birds-200-2011 Dataset.* 2011.
- Oikarinen et al. *Label-free Concept Bottleneck Models.* ICLR 2023.
- Yang et al. *Language in a Bottle: Language Model Guided Concept Bottleneck for Interpretable Image Classification.* CVPR 2023.
- Yuksekgonul et al. *Post-hoc Concept Bottleneck Models.* ICLR 2023.
